# 完整的動手指南：可在 Google Colab GPU 上訓練你的神經網路模型

**主要依據：老師指定的 [知乎文章](https://zhuanlan.zhihu.com/p/379291795)。**
本筆記本依文章的 **11 個步驟**安排，以 **MNIST 手寫數字、PyTorch 與兩層前饋神經網路**
示範 Colab GPU 訓練；並對照英文原文與作者提供的 notebook 確認程式與超參數。

核心實作為 **784 → 32 → 10、ReLU、SGD**：先用學習率 **0.5 訓練 5 輪**，
再沿用同一組模型權重，以 **0.1 訓練 5 輪**，不使用 momentum。
種子、完整測試集評估、最佳模型保存與圖表匯出是本專案另加的實驗管理功能。

請由上往下執行。所有訓練程式已包含於本檔，不必複製 repository 或遠端匯入原始碼。
本筆記本沒有預填執行成果；loss、準確率與耗時都由實際執行後產生。


## 步驟 1：在 Google Colab 啟用 GPU

1. 開啟 [Google Colab](https://colab.research.google.com/)，選擇「檔案 → 上傳筆記本」，
   上傳這份 `colab_gpu_training.ipynb`。
2. 選擇「執行階段 → 變更執行階段類型」，將硬體加速器設成 **GPU**，再連線。
   介面名稱可能隨語言或版本略有不同。
3. 從第一個程式格開始依序執行；變更硬體或重新連線後，需重新執行所有程式格。

原文發布時的 GPU 型號與時數描述屬當時環境。現在 Colab 的 GPU 型號、用量限制與連線時間會變動，
無法保證獲配某一型號或固定使用時數，請以 [Colab 官方 FAQ](https://research.google.com/colaboratory/faq.html#resource-limits) 為準。
本教學預設 `REQUIRE_GPU = True`，稍後會確認 CUDA 真正可用。
若未取得 GPU 而要先以 CPU 練習，可在參數格改成 `False`，允許自動回退至 CPU；
GPU 可用時仍會選擇 GPU。報告請註明實際裝置，CPU 驗證不能當作 GPU 實驗成果。


## 步驟 2：匯入套件

文章使用 PyTorch、torchvision 與 Matplotlib。下方保留 Colab 預裝的相容 **torch / torchvision**，
只補裝缺少的 NumPy 或 Matplotlib；不無條件重裝 PyTorch 或 CUDA。
隨後的共用匯入格也定義 `Config` 與環境紀錄工具，方便完整記錄這次實驗。


In [ ]:
import importlib.util
import subprocess
import sys

missing = [name for name in ("numpy", "matplotlib")
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
for name in ("torch", "torchvision"):
    if importlib.util.find_spec(name) is None:
        raise ModuleNotFoundError(
            f"缺少 {name}。請使用標準 Colab Python 執行階段；"
            "本機環境請依 README 安裝相容的 PyTorch / torchvision。"
        )
print("套件檢查完成；保留目前的 PyTorch / torchvision。")


In [ ]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
import csv
import json
import math
import platform
import random
import time

import numpy as np
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt


@dataclass
class Config:
    epochs: int = 10
    batch_size: int = 128
    val_batch_size: int = 256
    learning_rate: float = 0.5
    second_learning_rate: float = 0.1
    switch_epoch: int = 5
    hidden_size: int = 32
    seed: int = 42
    val_size: int = 10000
    num_workers: int = 0
    quick: bool = False
    require_gpu: bool = False

    def __post_init__(self):
        if min(self.epochs, self.batch_size, self.val_batch_size, self.hidden_size, self.switch_epoch) < 1:
            raise ValueError("epochs, batch sizes, hidden_size and switch_epoch must be positive")
        for rate in (self.learning_rate, self.second_learning_rate):
            if not math.isfinite(rate) or rate <= 0:
                raise ValueError("learning rates must be finite and positive")
        if self.val_size < 1 or self.num_workers < 0:
            raise ValueError("val_size must be positive; num_workers cannot be negative")
        if self.quick:
            self.epochs = 1


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def environment_info(device):
    return {
        "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": str(torch.__version__),
        "torchvision": str(torchvision.__version__),
        "numpy": str(np.__version__),
        "device": str(device),
        "cuda_available": torch.cuda.is_available(),
        "cuda_runtime": torch.version.cuda,
        "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "cpu_threads": torch.get_num_threads(),
    }


### 執行參數與成果位置

完整模式預設 **10 epochs、訓練 batch 128、驗證／測試 batch 256**。
第 1～5 輪使用 SGD 學習率 0.5，第 6～10 輪改成 0.1，承接原模型權重。
這對應作者 notebook 的兩次 `fit(5, ...)`；本專案以一次 `fit` 保存連續十輪的紀錄。

`QUICK = True` 會使用少量資料並強制只跑一輪，用於檢查流程，不能代表完整的兩階段實驗。
seed 42 是本專案新增的固定設定；不同硬體或套件版本仍可能出現數值差異。

Colab 暫存檔可能隨執行階段結束而清除。可把 `USE_GOOGLE_DRIVE` 改為 `True`，
依畫面授權掛載自己的 Drive，將成果寫入 `MyDrive/colab_gpu_training`。
每次執行參數格會建立新的時間戳記資料夾，避免混用舊 checkpoint。
重新實驗時，請從本格開始依序重跑。


In [ ]:
REQUIRE_GPU = True
QUICK = False
USE_GOOGLE_DRIVE = False
EPOCHS = 10

CONFIG = Config(
    epochs=EPOCHS,
    batch_size=128,
    val_batch_size=256,
    learning_rate=0.5,
    second_learning_rate=0.1,
    switch_epoch=5,
    hidden_size=32,
    seed=42,
    val_size=10000,
    num_workers=0,
    quick=QUICK,
    require_gpu=REQUIRE_GPU,
)
DATA_DIR = Path("data")
OUTPUT_ROOT = Path("outputs")
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/colab_gpu_training")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_ROOT / datetime.now(timezone.utc).strftime("colab_%Y%m%dT%H%M%S_%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)
seed_everything(CONFIG.seed)
print("參數：", asdict(CONFIG))
print("成果資料夾：", RUN_DIR.resolve())


## 步驟 3：準備 MNIST 並切分訓練／驗證資料

MNIST 每張影像是 **28 × 28 灰階手寫數字**，標籤為 0～9。
依文章作法，僅使用 `transforms.ToTensor()` 將像素轉成 0～1 張量，**不另加 Normalize**。
原始 60,000 張訓練影像分成 **50,000 張訓練、10,000 張驗證**。
本專案以 seed 42 固定隨機切分，讓各次實驗使用相同資料。

另外保留官方 **10,000 張測試影像**，不參與梯度更新、學習率調整或最佳模型挑選。
測試集是本專案額外加入的最終評估；文章核心練習主要觀察驗證表現。


In [ ]:
CLASS_NAMES = [str(digit) for digit in range(10)]


def split_indices(size, val_size, seed):
    if not 0 < val_size < size:
        raise ValueError("val_size must leave at least one training sample")
    order = torch.randperm(size, generator=torch.Generator().manual_seed(seed)).tolist()
    return order[val_size:], order[:val_size]


## 步驟 4：建立 DataLoader

DataLoader 將資料組成 mini-batch。訓練用 `batch_size=128` 並打亂順序，
驗證／測試用 `batch_size=256` 且不打亂。
下格第一次執行會下載 MNIST，之後重用 `DATA_DIR` 中的檔案。
此時 DataLoader 仍在 CPU；步驟 8 會用 `DeviceDataLoader` 包裝，將每個 batch 自動移到裝置。

預覽取自訓練集，影像仍是 `ToTensor()` 的 0～1 值，可直接顯示。
請確認實際切分數量與影像標籤；快速模式會顯示縮小後的資料量。


In [ ]:
def make_loaders(config, data_dir="data"):
    # The assigned article uses ToTensor only: pixels remain in [0, 1].
    transform = transforms.ToTensor()
    full_train = datasets.MNIST(data_dir, train=True, download=True, transform=transform)
    test_set = datasets.MNIST(data_dir, train=False, download=True, transform=transform)
    train_ids, val_ids = split_indices(len(full_train), config.val_size, config.seed)
    train_set = Subset(full_train, train_ids)
    val_set = Subset(full_train, val_ids)
    if config.quick:
        # Only for smoke testing, never report these as full-dataset results.
        train_set = Subset(train_set, range(min(2048, len(train_set))))
        val_set = Subset(val_set, range(min(512, len(val_set))))
        test_set = Subset(test_set, range(min(512, len(test_set))))
    options = {"num_workers": config.num_workers,
               "pin_memory": torch.cuda.is_available()}
    train_loader = DataLoader(train_set, batch_size=config.batch_size, shuffle=True,
                              generator=torch.Generator().manual_seed(config.seed), **options)
    return (train_loader, DataLoader(val_set, batch_size=config.val_batch_size, shuffle=False, **options),
            DataLoader(test_set, batch_size=config.val_batch_size, shuffle=False, **options))


In [ ]:
train_loader, val_loader, test_loader = make_loaders(CONFIG, data_dir=DATA_DIR)
split_sizes = {
    "train": len(train_loader.dataset),
    "validation": len(val_loader.dataset),
    "test": len(test_loader.dataset),
}
print("資料切分：", split_sizes)
fig, axes = plt.subplots(3, 4, figsize=(8, 6))
for index, axis in enumerate(axes.flat):
    image, label = train_loader.dataset[index]
    axis.imshow(image.squeeze(0).numpy(), cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"Label: {int(label)}")
    axis.axis("off")
fig.suptitle("MNIST training samples")
fig.tight_layout()
plt.show()
plt.close(fig)


## 步驟 5：定義 MnistModel

依文章與作者 notebook 使用兩層前饋神經網路：

| 階段 | 每張影像的維度 |
| --- | --- |
| 輸入灰階影像 | 1 × 28 × 28 |
| 展平成向量 | 784 |
| Linear 784→32，再經 ReLU | 32 |
| Linear 32→10，輸出 logits | 10 |

預設模型共有 **25,450 個參數**：`784×32 + 32 + 32×10 + 10`。
`CrossEntropyLoss` 直接接受 logits，模型尾端不需要加 Softmax。
下格先定義類別，步驟 10 才實際建立並移動模型。


In [ ]:
class MnistModel(nn.Module):
    def __init__(self, input_size=784, hidden_size=32, out_size=10):
        super().__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, out_size)

    def forward(self, images):
        # Return logits: CrossEntropyLoss applies log-softmax internally.
        flattened = images.reshape(images.size(0), -1)
        hidden = torch.relu(self.linear1(flattened))
        return self.linear2(hidden)


## 步驟 6：以 get_default_device 選擇裝置

文章的 `get_default_device()` 會在 CUDA 可用時選 GPU，否則回傳 CPU。
本專案另加 `get_device(require_gpu=True)` 檢查：若本次實驗要求 GPU 卻沒有 CUDA，就停止並明確提示。
這能避免把默默改用 CPU 的執行誤寫成 GPU 成果。

若下格停止，請確認步驟 1 的 GPU 設定並重新連線，或稍後再試。
若未取得 GPU 而要先以 CPU 練習，回到參數格將 `REQUIRE_GPU = False` 再依序重跑；
這會允許 CPU 備用，GPU 可用時仍會選擇 GPU。
只設定 Colab 選單不足以確認成功，應保留這格印出的實際裝置、GPU 名稱及套件版本。


In [ ]:
def get_default_device():
    """Choose CUDA when available, with the article's automatic CPU fallback."""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def get_device(require_gpu=False):
    device = get_default_device()
    if require_gpu and device.type != "cuda":
        raise RuntimeError(
            "CUDA GPU unavailable. In Colab choose Runtime > Change runtime type > GPU, "
            "then reconnect. To intentionally use CPU set REQUIRE_GPU=False "
            "or omit --require-gpu. GPU availability is not guaranteed."
        )
    return device


In [ ]:
print("get_default_device()：", get_default_device())
device = get_device(require_gpu=CONFIG.require_gpu)
environment = environment_info(device)
print(json.dumps(environment, ensure_ascii=False, indent=2))
(RUN_DIR / "config.json").write_text(json.dumps(asdict(CONFIG), ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_DIR / "environment.json").write_text(json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8")
print("本次模式：", "快速流程檢查" if CONFIG.quick else "完整兩階段實驗")


## 步驟 7：定義 to_device

資料和模型必須位於同一裝置。`to_device(data, device)` 會遞迴處理 list／tuple，
讓包含影像與標籤的 batch 一起移到 GPU 或 CPU，也能移動模型。
這把文章的裝置搬移概念整理為可重用工具，避免只移動模型、忘記移動資料。


In [ ]:
def to_device(data, device):
    """Move a tensor, model, or recursively nested list/tuple to one device."""
    if isinstance(data, list):
        return [to_device(item, device) for item in data]
    if isinstance(data, tuple):
        return tuple(to_device(item, device) for item in data)
    return data.to(device, non_blocking=True)


In [ ]:
example_tensor = torch.zeros(2, 1, 28, 28)
example_tensor = to_device(example_tensor, device)
print("範例張量所在裝置：", example_tensor.device)


## 步驟 8：以 DeviceDataLoader 自動搬移 batch

`DeviceDataLoader` 包裝原本的 DataLoader，在每次迭代時呼叫 `to_device`。
如此訓練與驗證迴圈收到的影像、標籤就已位於選定裝置。
下格明確包裝三個 loader，並檢查一個驗證 batch 的影像與標籤裝置。


In [ ]:
class DeviceDataLoader:
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device

    def __iter__(self):
        for batch in self.dl:
            yield to_device(batch, self.device)

    def __len__(self):
        return len(self.dl)

    @property
    def dataset(self):
        return self.dl.dataset


In [ ]:
train_loader = DeviceDataLoader(train_loader, device)
val_loader = DeviceDataLoader(val_loader, device)
test_loader = DeviceDataLoader(test_loader, device)
images, labels = next(iter(val_loader))
print("影像裝置：", images.device, "標籤裝置：", labels.device)
print("影像形狀：", tuple(images.shape), "標籤形狀：", tuple(labels.shape))


## 步驟 9：定義 fit 與 evaluate

`fit` 的 mini-batch 迴圈依序進行：**清除梯度 → 前向傳播 → 計算交叉熵 → 反向傳播 → SGD 更新**。
每輪切換 `model.train()`／`model.eval()`；`evaluate` 以停用梯度的模式計算驗證 loss 與 accuracy，
不更新權重。本專案依每個 batch 的樣本數加權平均 loss，讓最後一個較小 batch 的權重正確。

作者 notebook 先以 `fit(5, 0.5, ...)` 訓練，再沿用同一個 model 呼叫 `fit(5, 0.1, ...)`。
下方共用 `fit` 用連續十輪紀錄實作相同的 **5 輪 0.5 ＋ 5 輪 0.1**，
不重新初始化權重，SGD 不使用 momentum，並將每輪學習率存入 `history.csv`。
最低驗證 loss 的 checkpoint 保存是本專案新增功能，不使用測試集挑模型。


In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss, correct, count = 0.0, 0, 0
    for images, labels in loader:
        images, labels = to_device((images, labels), device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        count += labels.size(0)
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
    if count == 0:
        raise ValueError("Training loader must not be empty")
    return total_loss / count, correct / count


@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    total_loss, count = 0.0, 0
    labels_all, predictions_all = [], []
    for images, labels in loader:
        images, labels = to_device((images, labels), device)
        logits = model(images)
        total_loss += loss_fn(logits, labels).item()
        count += labels.size(0)
        labels_all.append(labels.cpu())
        predictions_all.append(logits.argmax(1).cpu())
    if count == 0:
        raise ValueError("Evaluation loader must not be empty")
    y_true, y_pred = torch.cat(labels_all), torch.cat(predictions_all)
    return {"loss": total_loss / count,
            "accuracy": (y_true == y_pred).sum().item() / count,
            "y_true": y_true, "y_pred": y_pred}


def fit(model, train_loader, val_loader, config, device, run_dir):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.SGD(model.parameters(), lr=config.learning_rate)
    history, best_loss, best_epoch = [], float("inf"), 0
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    started = time.perf_counter()
    for epoch in range(1, config.epochs + 1):
        epoch_started = time.perf_counter()
        if epoch == config.switch_epoch + 1:
            # Match the author's second fit(5, 0.1, ...): same weights, new plain SGD.
            optimizer = torch.optim.SGD(model.parameters(), lr=config.second_learning_rate)
        current_lr = optimizer.param_groups[0]["lr"]
        train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, device)
        validation = evaluate(model, val_loader, device)
        if not math.isfinite(train_loss) or not math.isfinite(validation["loss"]):
            raise RuntimeError("Non-finite loss. Check inputs and reduce the learning rate.")
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        row = {"epoch": epoch, "learning_rate": current_lr,
               "train_loss": train_loss, "train_accuracy": train_accuracy,
               "val_loss": validation["loss"], "val_accuracy": validation["accuracy"],
               "epoch_seconds": time.perf_counter() - epoch_started}
        history.append(row)
        if validation["loss"] < best_loss:
            best_loss, best_epoch = validation["loss"], epoch
            torch.save({"model_state_dict": model.state_dict(), "epoch": epoch,
                        "val_loss": best_loss, "config": asdict(config)},
                       run_dir / "best_model.pt")
        # Flush the learning history after every epoch so a disconnect loses less work.
        with (run_dir / "history.csv").open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(row))
            writer.writeheader()
            writer.writerows(history)
        print(f"Epoch {epoch:02d}/{config.epochs} | lr {current_lr:g} | loss {train_loss:.4f} | "
              f"acc {train_accuracy:.2%} | val_loss {validation['loss']:.4f} | "
              f"val_acc {validation['accuracy']:.2%}", flush=True)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    return history, best_epoch, time.perf_counter() - started


## 步驟 10：建立模型並移到選定裝置

現在建立 `MnistModel(784, 32, 10)`，並以 `to_device` 移到與 batch 相同的裝置。
建立前重新設定 seed，讓前面展示 batch 的操作不影響模型初始化。
預設參數量應為 **25,450**；形狀檢查中，`[2, 1, 28, 28]` 輸入應得到 `[2, 10]` logits。


In [ ]:
seed_everything(CONFIG.seed)
model = MnistModel(input_size=784, hidden_size=CONFIG.hidden_size, out_size=10)
model = to_device(model, device)
print(model)
parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"可訓練參數：{parameter_count:,}")
with torch.no_grad():
    example_input = torch.zeros(2, 1, 28, 28, device=device)
    example_output = model(example_input)
print("輸入形狀：", tuple(example_input.shape), "輸出形狀：", tuple(example_output.shape))
print("模型所在裝置：", next(model.parameters()).device)


## 步驟 11：先驗證未訓練模型，再訓練與驗證

依文章先觀察未訓練模型的驗證表現，接著才更新權重。
本專案把未訓練的 **loss／accuracy 純量**存為 `initial_validation.json`，方便和訓練後比較。
完整模式會顯示連續十輪紀錄，前五輪學習率為 0.5，後五輪為 0.1。
快速模式只檢查一輪，不涵蓋第二階段，不能用於聲稱完成文章的全部訓練。

請看驗證 loss 與 accuracy 是否改善；若沒有改善，先檢查資料、裝置與學習率，
不要用尚未評估的測試集調整參數。訓練結果由本次環境實測，不保證重現原文某個固定分數。


In [ ]:
initial = evaluate(model, val_loader, device)
initial_validation = {"loss": float(initial["loss"]), "accuracy": float(initial["accuracy"])}
(RUN_DIR / "initial_validation.json").write_text(
    json.dumps(initial_validation, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("未訓練模型的驗證表現：", initial_validation)
del initial


In [ ]:
history, best_epoch, train_seconds = fit(
    model, train_loader, val_loader, CONFIG, device, RUN_DIR
)
print(f"最低驗證 loss 位於第 {best_epoch} 輪；訓練耗時 {train_seconds:.2f} 秒。")


## 本專案補充：還原最佳模型並評估官方測試集

上面已完成文章的 11 個步驟。以下為本專案新增的實驗保存與分析功能。
最後一輪不一定具有最低驗證 loss，先還原本次實驗資料夾的最佳 checkpoint，
再對保留的官方測試集 **評估一次**；圖表與各類指標沿用同一份預測。

`map_location=device` 讓權重載入目前裝置；`weights_only=True` 用於自己剛保存的基本資料與張量 checkpoint。
此檔供模型推論使用，沒有保存可完整恢復訓練的 optimizer 狀態。

成果包括 `initial_validation.json`、`history.csv`、`metrics.json`、`environment.json`、`config.json`、
`best_model.pt`、`confusion_matrix.csv`，以及訓練曲線、混淆矩陣和預測圖；有錯誤時另存錯誤預測圖。
混淆矩陣的 **列是真實數字、欄是預測數字**。請同時閱讀 loss、accuracy、混淆矩陣與誤分類案例。


In [ ]:
checkpoint = torch.load(RUN_DIR / "best_model.pt", map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model = to_device(model, device)
model.eval()
print("已還原第", checkpoint["epoch"], "輪，驗證 loss =", checkpoint["val_loss"])


In [ ]:
def confusion_counts(y_true, y_pred, num_classes=10):
    counts = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(counts, (np.asarray(y_true), np.asarray(y_pred)), 1)
    return counts


def plot_history(history, path):
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for prefix, label in [("train", "Training"), ("val", "Validation")]:
        axes[0].plot(epochs, [row[prefix + "_loss"] for row in history], "o-", label=label)
        axes[1].plot(epochs, [row[prefix + "_accuracy"] for row in history], "o-", label=label)
    for axis, title in zip(axes, ["Cross-entropy loss", "Accuracy"]):
        axis.set(xlabel="Epoch", title=title, xticks=epochs)
        axis.grid(alpha=0.2)
        axis.legend()
    axes[1].set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_confusion(counts, path):
    fig, axis = plt.subplots(figsize=(10, 9))
    chart = axis.imshow(counts, cmap="Blues")
    fig.colorbar(chart, ax=axis, fraction=0.046, pad=0.04)
    axis.set(xticks=range(10), yticks=range(10), xticklabels=CLASS_NAMES,
             yticklabels=CLASS_NAMES, xlabel="Predicted class", ylabel="True class",
             title="Test confusion matrix (counts)")
    plt.setp(axis.get_xticklabels(), rotation=45, ha="right")
    for row in range(10):
        for col in range(10):
            axis.text(col, row, str(counts[row, col]), ha="center", va="center", fontsize=8,
                      color="white" if counts[row, col] > counts.max() / 2 else "black")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_predictions(dataset, y_true, y_pred, path, errors_only=False):
    indices = torch.where(y_true != y_pred)[0].tolist() if errors_only else list(range(len(y_true)))
    indices = indices[:12]
    if not indices:
        return False
    rows = math.ceil(len(indices) / 4)
    fig, axes = plt.subplots(rows, 4, figsize=(12, rows * 3), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for axis, index in zip(axes.flat, indices):
        image, _ = dataset[index]
        axis.imshow(image.squeeze().numpy().clip(0, 1), cmap="gray", vmin=0, vmax=1)
        true_id, predicted_id = y_true[index].item(), y_pred[index].item()
        axis.set_title(f"True: {CLASS_NAMES[true_id]}\nPred: {CLASS_NAMES[predicted_id]}",
                       color="green" if true_id == predicted_id else "crimson", fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return True


def save_artifacts(model, test_loader, history, config, device, run_dir,
                   best_epoch, train_seconds, split_sizes):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    result = evaluate(model, test_loader, device)
    counts = confusion_counts(result["y_true"], result["y_pred"])
    per_class = []
    for index, name in enumerate(CLASS_NAMES):
        true_positive = int(counts[index, index])
        predicted = int(counts[:, index].sum())
        support = int(counts[index].sum())
        precision = true_positive / predicted if predicted else 0.0
        recall = true_positive / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class.append({"class_id": index, "class_name": name, "precision": precision,
                          "recall": recall, "f1": f1, "support": support})
    metrics = {"dataset": "MNIST", "model": f"MLP 784-{config.hidden_size}-10", "optimizer": "SGD",
               "learning_rate_schedule": [{"epoch": row["epoch"], "lr": row["learning_rate"]} for row in history],
               "test_loss": result["loss"], "test_accuracy": result["accuracy"],
               "best_epoch": best_epoch,
               "best_val_loss": min(row["val_loss"] for row in history),
               "train_seconds": train_seconds,
               "parameter_count": sum(p.numel() for p in model.parameters()),
               "split_sizes": split_sizes, "quick_mode": config.quick,
               "per_class": per_class,
               "macro_f1": sum(row["f1"] for row in per_class) / len(per_class)}
    for name, payload in [("config.json", asdict(config)),
                          ("environment.json", environment_info(device)), ("metrics.json", metrics)]:
        (run_dir / name).write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    with (run_dir / "confusion_matrix.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["true_label"] + [f"pred_{label}" for label in CLASS_NAMES])
        for index, row in enumerate(counts):
            writer.writerow([index] + row.tolist())
    plot_history(history, run_dir / "training_curves.png")
    plot_confusion(counts, run_dir / "confusion_matrix.png")
    plot_predictions(test_loader.dataset, result["y_true"], result["y_pred"], run_dir / "predictions.png")
    plot_predictions(test_loader.dataset, result["y_true"], result["y_pred"],
                     run_dir / "misclassified.png", errors_only=True)
    print(f"Best epoch: {best_epoch} | Test accuracy: {result['accuracy']:.2%}", flush=True)
    return metrics


In [ ]:
metrics = save_artifacts(
    model, test_loader, history, CONFIG, device, RUN_DIR,
    best_epoch, train_seconds, split_sizes,
)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
try:
    from IPython.display import Image, display
except ImportError:
    Image = None
for filename in ("training_curves.png", "confusion_matrix.png", "predictions.png", "misclassified.png"):
    image_path = RUN_DIR / filename
    if image_path.exists():
        if Image is not None:
            display(Image(filename=str(image_path), width=900))
        else:
            print("圖表：", image_path.resolve())
print("成果檔案：", [path.name for path in sorted(RUN_DIR.iterdir())])


### 載入權重進行單張推論

新建相同結構的模型並載入權重，即可恢復模型做推論。
下格從已完成評估的測試資料取一張影像，以 `unsqueeze(0)` 加上 batch 維度，
再用 `argmax` 取得預測數字。這是單張推論示範，不重新計算測試指標。
預測錯誤也應如實保留；單張成功案例不代表整體準確率。


In [ ]:
reloaded_model = MnistModel(input_size=784, hidden_size=CONFIG.hidden_size, out_size=10)
reloaded_model = to_device(reloaded_model, device)
saved_checkpoint = torch.load(RUN_DIR / "best_model.pt", map_location=device, weights_only=True)
reloaded_model.load_state_dict(saved_checkpoint["model_state_dict"])
reloaded_model.eval()
sample_image, sample_label = test_loader.dataset[0]
with torch.no_grad():
    sample_logits = reloaded_model(to_device(sample_image.unsqueeze(0), device))
    sample_prediction = int(sample_logits.argmax(dim=1).item())
print("真實數字：", int(sample_label))
print("預測數字：", sample_prediction)


### 保存與下載實驗成果

若使用 Colab 暫存磁碟，離開前可把 `DOWNLOAD_RESULTS` 改成 `True`，壓縮並下載本次成果。
Drive 使用者也可在自己的 Drive 取回檔案。
報告應記錄實際資料量、seed、兩階段學習率、epochs、未訓練驗證指標、最佳輪次、測試指標、耗時及實際 GPU 型號。
本機 CPU 驗證、快速模式和完整 Colab GPU 實驗需分開標示。

在 Colab 選「檔案 → 下載 → 下載 .ipynb」可保存執行輸出，再將修訂後的 notebook 上傳自己的 GitHub。
分享前請檢查輸出是否包含私人 Drive 路徑或其他個人資訊。


In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files
    archive_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
    files.download(archive_path)
else:
    print("成果保存在：", RUN_DIR.resolve())


## 依據、對照與延伸

- **老師指定的主要依據：**[知乎文章](https://zhuanlan.zhihu.com/p/379291795)。本筆記本的 11 步驟順序依此安排。
- [英文原文：A Complete Hands-on Guide to Train Your Neural Network Model on Google Colab GPU](https://www.analyticsvidhya.com/blog/2021/05/a-complete-hands-on-guide-to-train-your-neural-network-model-on-google-colab-gpu/)
- [作者 notebook：兩層 MnistModel 與 5 輪 0.5 ＋ 5 輪 0.1 SGD](https://github.com/radathan1/Running-Deep-Learning-Model-on-GPU/blob/main/Running_Deep_Learning_Model_on_GPU%20%281%29.ipynb)
- [Google Colab 官方 FAQ：當前 GPU 與資源限制](https://research.google.com/colaboratory/faq.html)
- [torchvision 官方 MNIST 文件](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html)
- [PyTorch 官方 torch.load 文件](https://docs.pytorch.org/docs/stable/generated/torch.load.html)

文章中的排版與程式片段語法問題已整理為可依序執行的 Python；核心資料、模型、裝置搬移與兩階段 SGD 依作者 notebook。
seed 42、樣本加權平均、停用梯度的評估、官方測試集、checkpoint 與成果匯出屬本專案補充。
GitHub／本機專案另附 [文章對照說明](../docs/SOURCE_ALIGNMENT.md)，逐項區分來源核心與新增功能。
延伸實驗請每次只改一項設定，以驗證集比較，並分清實際觀察與待驗證的假設。
